```python
1. Change column names:
   + dr.rename(): new_name = old_name
   + dr.rename_with() and lambda
   + Apply pandas method with dr.pipe()

2. Change row names (index): use dr.mutate()
```

In [37]:
import datar.all as dr
from datar import f
import polars as pl
from pathlib import Path

from pipda import register_verb
import warnings

dr.filter = register_verb(func=dr.filter_)
warnings.filterwarnings("ignore")
pl.Config.set_tbl_cell_alignment("CENTER")

data_dir = Path("/home").rglob("*/DataScience_MachineLearning/data")
data_dir = next(data_dir)

In [38]:
tb_emp = dr.tibble(pl.read_csv(data_dir/"emp.csv"))
print(tb_emp)

shape: (8, 5)
┌─────┬──────────┬────────┬────────────┬────────────┐
│  id ┆   name   ┆ salary ┆ start_date ┆    dept    │
│ --- ┆    ---   ┆   ---  ┆     ---    ┆     ---    │
│ i64 ┆    str   ┆   f64  ┆     str    ┆     str    │
╞═════╪══════════╪════════╪════════════╪════════════╡
│  1  ┆   Rick   ┆  623.3 ┆ 2012-01-01 ┆     IT     │
│  2  ┆    Dan   ┆  515.2 ┆ 2013-09-23 ┆ Operations │
│  3  ┆ Michelle ┆  611.0 ┆ 2014-11-15 ┆     IT     │
│  4  ┆   Ryan   ┆  729.0 ┆ 2014-05-11 ┆     HR     │
│  5  ┆   Gary   ┆ 843.25 ┆ 2015-03-27 ┆   Finance  │
│  6  ┆   Nina   ┆  578.0 ┆ 2013-05-21 ┆     IT     │
│  7  ┆   Simon  ┆  632.8 ┆ 2013-07-30 ┆ Operations │
│  8  ┆   Guru   ┆  722.5 ┆ 2014-06-17 ┆   Finance  │
└─────┴──────────┴────────┴────────────┴────────────┘


# <span style="color:#1E90FF">1. Change column names</span>
## 1.1. dr.rename(): new_name = old_name

In [39]:
tb_renamed = (
    tb_emp
    >> dr.rename(
        emp_id = f.id, 
        emp_name = f.name
    )
)

print(tb_renamed.head())

shape: (5, 5)
┌────────┬──────────┬────────┬────────────┬────────────┐
│ emp_id ┆ emp_name ┆ salary ┆ start_date ┆    dept    │
│   ---  ┆    ---   ┆   ---  ┆     ---    ┆     ---    │
│   i64  ┆    str   ┆   f64  ┆     str    ┆     str    │
╞════════╪══════════╪════════╪════════════╪════════════╡
│    1   ┆   Rick   ┆  623.3 ┆ 2012-01-01 ┆     IT     │
│    2   ┆    Dan   ┆  515.2 ┆ 2013-09-23 ┆ Operations │
│    3   ┆ Michelle ┆  611.0 ┆ 2014-11-15 ┆     IT     │
│    4   ┆   Ryan   ┆  729.0 ┆ 2014-05-11 ┆     HR     │
│    5   ┆   Gary   ┆ 843.25 ┆ 2015-03-27 ┆   Finance  │
└────────┴──────────┴────────┴────────────┴────────────┘


## 1.2. dr.rename_with() and lambda

In [40]:
# Use with python built in function ``str.upper``

tb_renamed2 = (
    tb_emp
    >> dr.rename_with(str.upper, f[f.salary, f.dept])
)

print(tb_renamed2.tail())

shape: (5, 5)
┌─────┬───────┬────────┬────────────┬────────────┐
│  id ┆  name ┆ SALARY ┆ start_date ┆    DEPT    │
│ --- ┆  ---  ┆   ---  ┆     ---    ┆     ---    │
│ i64 ┆  str  ┆   f64  ┆     str    ┆     str    │
╞═════╪═══════╪════════╪════════════╪════════════╡
│  4  ┆  Ryan ┆  729.0 ┆ 2014-05-11 ┆     HR     │
│  5  ┆  Gary ┆ 843.25 ┆ 2015-03-27 ┆   Finance  │
│  6  ┆  Nina ┆  578.0 ┆ 2013-05-21 ┆     IT     │
│  7  ┆ Simon ┆  632.8 ┆ 2013-07-30 ┆ Operations │
│  8  ┆  Guru ┆  722.5 ┆ 2014-06-17 ┆   Finance  │
└─────┴───────┴────────┴────────────┴────────────┘


In [41]:
# Use with lambda function

tb_pokemon = (
    dr.tibble(pl.read_csv(data_dir/"pokemon.csv"))
    >> dr.select(~f["#"])
    >> dr.rename_with(lambda col: col.strip().replace(" ", "_").replace(".", ""))
)

print(tb_pokemon.head())

shape: (5, 12)
┌───────────────────────┬────────┬────────┬───────┬───┬────────┬───────┬────────────┬───────────┐
│          Name         ┆ Type_1 ┆ Type_2 ┆ Total ┆ … ┆ Sp_Def ┆ Speed ┆ Generation ┆ Legendary │
│          ---          ┆   ---  ┆   ---  ┆  ---  ┆   ┆   ---  ┆  ---  ┆     ---    ┆    ---    │
│          str          ┆   str  ┆   str  ┆  i64  ┆   ┆   i64  ┆  i64  ┆     i64    ┆    bool   │
╞═══════════════════════╪════════╪════════╪═══════╪═══╪════════╪═══════╪════════════╪═══════════╡
│       Bulbasaur       ┆  Grass ┆ Poison ┆  318  ┆ … ┆   65   ┆   45  ┆      1     ┆   false   │
│        Ivysaur        ┆  Grass ┆ Poison ┆  405  ┆ … ┆   80   ┆   60  ┆      1     ┆   false   │
│        Venusaur       ┆  Grass ┆ Poison ┆  525  ┆ … ┆   100  ┆   80  ┆      1     ┆   false   │
│ VenusaurMega Venusaur ┆  Grass ┆ Poison ┆  625  ┆ … ┆   120  ┆   80  ┆      1     ┆   false   │
│       Charmander      ┆  Fire  ┆  null  ┆  309  ┆ … ┆   50   ┆   65  ┆      1     ┆   false   │
└────

## 1.3. Apply polars method with dr.pipe()

In [42]:
tb_renamed3 = (
    tb_emp
    >> dr.pipe(lambda f: f.select(pl.all().name.prefix('emp_')))
    >> dr.filter(f.emp_salary > 600)
)

print(tb_renamed3.head())

shape: (5, 5)
┌────────┬──────────┬────────────┬────────────────┬────────────┐
│ emp_id ┆ emp_name ┆ emp_salary ┆ emp_start_date ┆  emp_dept  │
│   ---  ┆    ---   ┆     ---    ┆       ---      ┆     ---    │
│   i64  ┆    str   ┆     f64    ┆       str      ┆     str    │
╞════════╪══════════╪════════════╪════════════════╪════════════╡
│    1   ┆   Rick   ┆    623.3   ┆   2012-01-01   ┆     IT     │
│    3   ┆ Michelle ┆    611.0   ┆   2014-11-15   ┆     IT     │
│    4   ┆   Ryan   ┆    729.0   ┆   2014-05-11   ┆     HR     │
│    5   ┆   Gary   ┆   843.25   ┆   2015-03-27   ┆   Finance  │
│    7   ┆   Simon  ┆    632.8   ┆   2013-07-30   ┆ Operations │
└────────┴──────────┴────────────┴────────────────┴────────────┘


# <span style="color:#1E90FF">2. Change row names (index): use dr.mutate()</span>

In [43]:
tb_rownames = (
    tb_emp
    >> dr.mutate(id = [f"employee_{i}" for i in range(1, len(tb_emp) + 1)])
)

print(tb_rownames.head())

shape: (5, 5)
┌────────────┬──────────┬────────┬────────────┬────────────┐
│     id     ┆   name   ┆ salary ┆ start_date ┆    dept    │
│     ---    ┆    ---   ┆   ---  ┆     ---    ┆     ---    │
│     str    ┆    str   ┆   f64  ┆     str    ┆     str    │
╞════════════╪══════════╪════════╪════════════╪════════════╡
│ employee_1 ┆   Rick   ┆  623.3 ┆ 2012-01-01 ┆     IT     │
│ employee_2 ┆    Dan   ┆  515.2 ┆ 2013-09-23 ┆ Operations │
│ employee_3 ┆ Michelle ┆  611.0 ┆ 2014-11-15 ┆     IT     │
│ employee_4 ┆   Ryan   ┆  729.0 ┆ 2014-05-11 ┆     HR     │
│ employee_5 ┆   Gary   ┆ 843.25 ┆ 2015-03-27 ┆   Finance  │
└────────────┴──────────┴────────┴────────────┴────────────┘
